# Задание

1. Найти датасет на hugging face, который будет содержать в себе числовые данные (> 3 числовых признаков)

2. Очистить данные от выбросов методами, рассмотренными на практическом занятии

3. Произвести отбор признаков одним из способов, рассмотренных на лекционной части занятия

# Импорт библиотек

Загружаем необходимые библиотеки для обработки данных, визуализации и машинного обучения.

In [35]:
%pip install pandas numpy plotly nbformat scikit-learn datasets


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [36]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from datasets import load_dataset

# Загрузка датасета

Загружаем датасет SimpleLinearRegression с Hugging Face, содержащий числовые признаки для регрессионной задачи.

In [37]:
dataset = load_dataset("Darkester/SimpleLinearRegression")
df = dataset["train"].to_pandas()

print(f"Shape: {df.shape}")
df.columns

Shape: (549, 7)


Index(['feature1', 'feature2', 'feature3', 'feature4', 'feature5', 'feature6',
       'target'],
      dtype='object')

In [38]:
df.head()

,feature1,feature2,feature3,feature4,feature5,feature6,target
0,5.2,3.1,7.8,2.5,4.0,6.3,38.7
1,6.0,2.8,8.2,3.0,3.5,5.9,39.1
2,4.8,3.5,7.5,2.8,4.2,6.1,37.3
3,5.5,3.0,8.0,2.7,3.8,6.0,38.5
4,6.2,2.9,7.9,3.1,3.7,5.8,39.3


In [39]:
df.describe()

,feature1,feature2,feature3,feature4,feature5,feature6,target
count,5.490000e+02,5.490000e+02,5.490000e+02,5.490000e+02,5.490000e+02,5.490000e+02,5.490000e+02
mean,2.936597e+05,3.694217e+05,6.252158e+05,4.025686e+05,4.972993e+05,7.171432e+05,1.841587e+06
std,9.586300e+05,1.195198e+06,1.958390e+06,1.268375e+06,1.556273e+06,2.264903e+06,5.755052e+06
min,-5.000000e-02,-1.800000e-01,4.000000e-02,-9.000000e-02,-1.400000e-01,-4.000000e-02,7.000000e-02
25%,7.300000e+00,3.300000e+00,8.200000e+00,3.300000e+00,4.100000e+00,5.100000e+00,6.470000e+01
50%,1.610000e+01,1.550000e+01,2.510000e+01,1.390000e+01,2.000000e+01,2.130000e+01,9.930000e+01
75%,7.500000e+01,4.500000e+01,1.100000e+02,2.680000e+01,6.000000e+01,4.370000e+01,4.000000e+02
max,5.000000e+06,6.000000e+06,8.500000e+06,5.700000e+06,6.600000e+06,1.010000e+07,2.425400e+07


# Обнаружение и удаление выбросов

Используем метод IQR (Interquartile Range) для выявления и удаления выбросов. Визуализируем распределение данных до и после обработки.

In [40]:
df_cleaned = df.copy()

outlier_bounds = {}
for col in df.columns[:-1]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_bounds[col] = (lower, upper)
    
    df_cleaned = df_cleaned[(df_cleaned[col] >= lower) & (df_cleaned[col] <= upper)]

fig = make_subplots(
    rows=len(df.columns) - 1, cols=2,
    subplot_titles=[f"{col} - Before" if i % 2 == 0 else f"{col} - After" 
                    for i in range(2 * (len(df.columns) - 1))]
)

for idx, col in enumerate(df.columns[:-1]):
    fig.add_trace(
        go.Box(y=df[col], name=col, showlegend=False),
        row=idx + 1, col=1
    )
    
    fig.add_trace(
        go.Box(y=df_cleaned[col], name=col, showlegend=False),
        row=idx + 1, col=2
    )
    
    print(f"{col}:")
    print(f"  Before: {df[col].count()} rows, mean={df[col].mean():.2f}, std={df[col].std():.2f}")
    print(f"  After:  {df_cleaned[col].count()} rows, mean={df_cleaned[col].mean():.2f}, std={df_cleaned[col].std():.2f}\n")

fig.update_layout(height=300 * (len(df.columns) - 1), title_text="Outlier Detection: IQR Method")
fig.show()

print(f"Total rows removed: {len(df) - len(df_cleaned)} ({(1 - len(df_cleaned)/len(df))*100:.1f}%)")

feature1:
  Before: 549 rows, mean=293659.74, std=958630.04
  After:  390 rows, mean=18.48, std=20.36

feature2:
  Before: 549 rows, mean=369421.74, std=1195197.69
  After:  390 rows, mean=15.48, std=16.34

feature3:
  Before: 549 rows, mean=625215.82, std=1958389.57
  After:  390 rows, mean=32.81, std=43.78

feature4:
  Before: 549 rows, mean=402568.65, std=1268374.61
  After:  390 rows, mean=12.02, std=11.20

feature5:
  Before: 549 rows, mean=497299.25, std=1556273.33
  After:  390 rows, mean=18.05, std=17.23

feature6:
  Before: 549 rows, mean=717143.24, std=2264902.94
  After:  390 rows, mean=18.56, std=18.05



Total rows removed: 159 (29.0%)


# Отбор признаков

Применяем два метода для выбора наиболее информативных признаков: Linear Regression коэффициенты и Random Forest важность. Сравниваем результаты.

## Подготовка данных

Разделяем данные на признаки (X) и целевую переменную (y).

In [41]:
X = df_cleaned.iloc[:, :-1]
y = df_cleaned.iloc[:, -1]

print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
print(f"Target: {y.name}")
print(f"\n{y.describe()}")

Features: 6, Samples: 390
Target: target

count     390.000000
mean      164.967974
std       364.150967
min         0.070000
25%        37.125000
50%        67.300000
75%       100.500000
max      2250.000000
Name: target, dtype: float64


In [42]:
lr = LinearRegression()
lr.fit(X, y)

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': np.abs(lr.coef_)
}).sort_values('Coefficient', ascending=False)
print("Linear Regression Coefficients:")
print(coefficients, "\n")
top_features_lr = coefficients['Feature'].head(5).tolist()

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)
print("Random Forest Importance:")
print(importance)
top_features_rf = importance['Feature'].head(5).tolist()

Linear Regression Coefficients:
    Feature  Coefficient
3  feature4    11.767188
2  feature3     6.309390
4  feature5     5.833783
5  feature6     2.779596
0  feature1     1.971105
1  feature2     0.094955 

Random Forest Importance:
    Feature  Importance
2  feature3    0.898711
5  feature6    0.042964
0  feature1    0.019365
3  feature4    0.018977
1  feature2    0.010033
4  feature5    0.009950


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Linear Regression", "Random Forest")
)

fig.add_trace(
    go.Bar(y=coefficients.head(10)['Feature'], x=coefficients.head(10)['Coefficient'], 
           orientation='h', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(y=importance.head(10)['Feature'], x=importance.head(10)['Importance'], 
           orientation='h', marker_color='green'),
    row=1, col=2
)

fig.update_xaxes(title_text="Coefficient", row=1, col=1)
fig.update_xaxes(title_text="Importance", row=1, col=2)
fig.update_layout(height=500, title_text="Feature Selection: Linear Regression vs Random Forest", showlegend=False)
fig.show()

final_features = list(set([f for f in top_features_lr if f in top_features_rf]))
if not final_features:
    final_features = sorted(set(top_features_lr + top_features_rf))[:5]

print(f"\nSelected features: {sorted(final_features)}")


Selected features: ['feature1', 'feature3', 'feature4', 'feature6']
